<a href="https://colab.research.google.com/github/xwiz/db-claw/blob/main/notebooks/phase_d_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase D â€” GPU Retrain on Colab

End-to-end pipeline: clone repo â†’ ingest HF corpora â†’ derive Stage 1+3 corpora â†’
train Stage 2 (t5-base) + Stage 1 + Stage 3 â†’ export cascade â†’ download.

Acceptance gate: BIRD-100 EX â‰¥ 25%. Per `docs/results/phase-d-gpu-handoff.md`.

## Runtime requirements

- **Colab T4** (free tier) for Stage 1 + Stage 3 (each ~30-60 min).
- **Colab A100 / Kaggle T4 Ã—2** for full Stage 2 t5-base Ã— 5 epochs (~12-24 hr).
- A100 fits t5-base bs=8 grad_accum=16 fp16 comfortably.
- T4 fits t5-small Ã— 5 epochs in ~6-8 hr â€” use as fallback.

## What this notebook ships

1. Clone the public repo.
2. Install ML extras (transformers, datasets, optimum, etc).
3. Stream raw corpora from HuggingFace:
   - `trl-lab/SQaLe-text-to-SQL-dataset` (~511K rows)
   - `NumbersStation/NSText2SQL` (~289K rows)
   - `gretelai/synthetic_text_to_sql` (~100K rows)
   - `xxxbrem/OmniSQL-BIRD` (9K rows)
   - Spider+BIRD dev manifests (uploaded as data fixtures)
4. Run `build-teacher-cache` per source â†’ ultimate v3 corpus (~500K rows).
5. Run `derive-linker-pairs` + `derive-slot-pairs` for Stage 1 + Stage 3.
6. Train all three stages.
7. Export cascade-v3-gpu + run BIRD smoke.
8. Tarball cascade artefacts â†’ upload to Drive / GitHub Releases.

## 1. Clone repo + install

In [14]:
!git clone https://github.com/xwiz/db-claw.git
%cd db-claw
!git log --oneline | head -5

Cloning into 'db-claw'...
remote: Enumerating objects: 392, done.
remote: Counting objects: 100% (392/392), done.
remote: Compressing objects: 100% (283/283), done.
remote: Total 392 (delta 100), reused 378 (delta 86), pack-reused 0 (from 0)
Receiving objects: 100% (392/392), 780.03 KiB | 12.19 MiB/s, done.
Resolving deltas: 100% (100/100), done.
/content/db-claw/db-claw
7b1ea4c Fix Colab notebook ingest crash + add Gretel column override flags
b15454f Push progress
0e19f1b Initial commit


In [15]:
# Install Python deps. Colab pre-installs torch + cuda; we add the
# rest via the package's own pyproject.
!pip install -q -e python/semsql_train python/semsql_eval python/semsql_rewriter
!pip install -q transformers==4.45.2 datasets accelerate optimum[onnxruntime] sqlglot click pyarrow huggingface_hub sentencepiece onnxruntime
!python -c 'import torch; print(f"torch={torch.__version__} cuda={torch.cuda.is_available()} dev={torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}")'

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for semsql-train (pyproject.toml) ... done
torch=2.10.0+cpu cuda=False dev=None


## 2. Fetch raw corpora

In [16]:
from huggingface_hub import hf_hub_download, snapshot_download
import shutil, os, glob
os.makedirs('data', exist_ok=True)

# SQaLe â€” full snapshot (33 parquet shards, ~7GB total)
snapshot_download(
    repo_id='trl-lab/SQaLe-text-to-SQL-dataset',
    repo_type='dataset',
    local_dir='data/sqale',
    allow_patterns=['data/*.parquet'],
)

# NSText2SQL
p = hf_hub_download(
    repo_id='NumbersStation/NSText2SQL',
    filename='train.jsonl',
    repo_type='dataset',
)
shutil.copy(p, 'data/nstext2sql_train.jsonl')

# Gretel synthetic
p = hf_hub_download(
    repo_id='gretelai/synthetic_text_to_sql',
    filename='synthetic_text_to_sql_train.snappy.parquet',
    repo_type='dataset',
)
shutil.copy(p, 'data/gretel_synth_text_to_sql_train.parquet')

# OmniSQL-BIRD
p = hf_hub_download(
    repo_id='xxxbrem/OmniSQL-BIRD',
    filename='train_bird.json',
    repo_type='dataset',
)
shutil.copy(p, 'data/omnisql_bird_train.json')

!ls -la data/

Fetching 33 files:   0%|          | 0/33 [00:00<?, ?it/s]

data/train-00003-of-00033.parquet:   0%|          | 0.00/110M [00:00<?, ?B/s]

data/train-00004-of-00033.parquet:   0%|          | 0.00/110M [00:00<?, ?B/s]

data/train-00006-of-00033.parquet:   0%|          | 0.00/110M [00:00<?, ?B/s]

data/train-00001-of-00033.parquet:   0%|          | 0.00/110M [00:00<?, ?B/s]

...

data/train-00029-of-00033.parquet:   0%|          | 0.00/109M [00:00<?, ?B/s]

data/train-00030-of-00033.parquet:   0%|          | 0.00/110M [00:00<?, ?B/s]

data/train-00031-of-00033.parquet:   0%|          | 0.00/110M [00:00<?, ?B/s]

data/train-00032-of-00033.parquet:   0%|          | 0.00/109M [00:00<?, ?B/s]

total 532224
drwxr-xr-x  3 root root      4096 May  9 05:47 .
drwxr-xr-x 12 root root      4096 May  9 05:46 ..
-rw-r--r--  1 root root  32363736 May  9 05:47 gretel_synth_text_to_sql_train.parquet
-rw-r--r--  1 root root 432924555 May  9 05:47 nstext2sql_train.jsonl
-rw-r--r--  1 root root  79688021 May  9 05:47 omnisql_bird_train.json
drwxr-xr-x  4 root root      4096 May  9 05:47 sqale


In [17]:
# Spider + BIRD dev manifests for eval. Upload via Colab UI (Files
# panel) or fetch from your own bucket. The repo doesn't ship them
# because they're licensed datasets.
from google.colab import files
print('Upload spider/dev.json + bird/dev.json as needed for the BIRD eval at the end.')
# uploaded = files.upload()  # uncomment when ready

Upload spider/dev.json + bird/dev.json as needed for the BIRD eval at the end.


## 3. Build ultimate v3 corpus (~10-15 min on Colab)

In [18]:
import sys
import os
import importlib.util
from pathlib import Path

# Direct path to the file we need
module_path = '/content/db-claw/python/semsql_train/src/semsql_train/teacher_cache.py'

# Manual module loading
spec = importlib.util.spec_from_file_location("semsql_train.teacher_cache", module_path)
tc_module = importlib.util.module_from_spec(spec)
sys.modules["semsql_train.teacher_cache"] = tc_module
spec.loader.exec_module(tc_module)

# Extract functions
build_teacher_cache_from_sqale = tc_module.build_teacher_cache_from_sqale
build_teacher_cache_from_nstext2sql_jsonl = tc_module.build_teacher_cache_from_nstext2sql_jsonl
build_teacher_cache_from_omnisql = tc_module.build_teacher_cache_from_omnisql
build_teacher_cache_from_omnisql_bird_json = tc_module.build_teacher_cache_from_omnisql_bird_json

print("Successfully loaded teacher_cache via importlib.")

import glob
sqale_files = glob.glob('data/sqale/data/*.parquet')
sqale_glob = 'data/sqale/data/*.parquet' if not sqale_files else sorted(sqale_files)[0].rsplit('/', 1)[0] + '/*.parquet'

s1 = build_teacher_cache_from_sqale(
    out_jsonl=Path('data/skeleton_train_v3_full.jsonl'),
    parquet_glob=sqale_glob,
)
print(f'sqale: {s1.converted}/{s1.total} retention={s1.retention:.1%}')

Successfully loaded teacher_cache via importlib.
[sqale] total=5000 converted=1885 retention=37.7%
[sqale] total=15000 converted=5550 retention=37.0%


[sqale] total=45000 converted=16660 retention=37.0%
[sqale] total=50000 converted=18471 retention=36.9%
[sqale] total=55000 converted=20372 retention=37.0%
[sqale] total=65000 converted=23977 retention=36.9%
[sqale] total=70000 converted=25781 retention=36.8%


[sqale] total=85000 converted=31187 retention=36.7%


[sqale] total=120000 converted=44062 retention=36.7%
[sqale] total=125000 converted=45888 retention=36.7%
[sqale] total=130000 converted=47683 retention=36.7%


[sqale] total=140000 converted=51327 retention=36.7%
[sqale] total=145000 converted=53162 retention=36.7%


...

[sqale] total=375000 converted=137306 retention=36.6%
[sqale] total=380000 converted=139115 retention=36.6%


[sqale] total=410000 converted=150135 retention=36.6%


[sqale] total=415000 converted=151941 retention=36.6%
[sqale] total=425000 converted=155621 retention=36.6%
[sqale] total=435000 converted=159371 retention=36.6%
[sqale] total=440000 converted=161175 retention=36.6%


[sqale] total=465000 converted=170409 retention=36.6%


[sqale] total=485000 converted=177792 retention=36.7%
[sqale] total=490000 converted=179545 retention=36.6%
[sqale] total=505000 converted=185016 retention=36.6%
sqale: 187400/511630 retention=36.6%


In [19]:
!ls -R /content/db-claw/python/semsql_train

/content/db-claw/python/semsql_train:
conftest.py  pyproject.toml  README.md	src  tests

/content/db-claw/python/semsql_train/src:
semsql_train

/content/db-claw/python/semsql_train/src/semsql_train:
active_subset.py	   __init__.py	   spider_linker.py
generators_linker_v3.py    __main__.py	   teacher_cache.py
generators.py		   onnx_export.py  templates.py
generators_slot_v3.py	   paraphrase.py   trainers
generators_targeted_v3.py  __pycache__

/content/db-claw/python/semsql_train/src/semsql_train/__pycache__:
teacher_cache.cpython-312.pyc

/content/db-claw/python/semsql_train/src/semsql_train/trainers:
distill.py  __init__.py  linker.py  skeleton.py  slot_filler.py

/content/db-claw/python/semsql_train/tests:
fixtures	       test_onnx_export.py    test_trainer_linker.py
__init__.py	       test_paraphrase.py     test_trainer_skeleton.py
test_active_subset.py  test_spider_linker.py  test_trainer_slot_filler.py
test_distill.py        test_teacher_cache.py
test_generators.py     test_templ

In [20]:
!git pull

Already up to date.


In [21]:
s2 = build_teacher_cache_from_nstext2sql_jsonl(
    in_jsonl=Path('data/nstext2sql_train.jsonl'),
    out_jsonl=Path('data/skeleton_train_v3_nstext2sql.jsonl'),
)
print(f'nstext2sql: {s2.converted}/{s2.total} retention={s2.retention:.1%}')

s3 = build_teacher_cache_from_omnisql(
    out_jsonl=Path('data/skeleton_train_v3_gretel.jsonl'),
    parquet_glob='data/gretel_synth_text_to_sql_train.parquet',
    question_key='sql_prompt',
    sql_key='sql',
    db_id_key='domain',
)
print(f'gretel: {s3.converted}/{s3.total} retention={s3.retention:.1%}')

s4 = build_teacher_cache_from_omnisql_bird_json(
    in_json=Path('data/omnisql_bird_train.json'),
    out_jsonl=Path('data/skeleton_train_v3_omnisql_bird.jsonl'),
)
print(f'omnisql-bird: {s4.converted}/{s4.total} retention={s4.retention:.1%}')

[nstext2sql] total=10000 converted=8001 retention=80.0%
[nstext2sql] total=15000 converted=12014 retention=80.1%
[nstext2sql] total=20000 converted=16056 retention=80.3%
[nstext2sql] total=25000 converted=20089 retention=80.4%
[nstext2sql] total=35000 converted=28187 retention=80.5%
[nstext2sql] total=40000 converted=32229 retention=80.6%
[nstext2sql] total=45000 converted=36303 retention=80.7%
[nstext2sql] total=50000 converted=40330 retention=80.7%
[nstext2sql] total=55000 converted=44321 retention=80.6%
[nstext2sql] total=60000 converted=48364 retention=80.6%
[nstext2sql] total=65000 converted=52409 retention=80.6%
[nstext2sql] total=70000 converted=56382 retention=80.5%
[nstext2sql] total=75000 converted=60359 retention=80.5%
[nstext2sql] total=80000 converted=64386 retention=80.5%
[nstext2sql] total=85000 converted=68399 retention=80.5%
[nstext2sql] total=90000 converted=72440 retention=80.5%
[nstext2sql] total=95000 converted=76484 retention=80.5%
[nstext2sql] total=100000 conver

[omnisql] total=20000 converted=13803 retention=69.0%


[omnisql] total=35000 converted=24208 retention=69.2%


[omnisql] total=40000 converted=27637 retention=69.1%


[omnisql] total=45000 converted=31091 retention=69.1%
[omnisql] total=50000 converted=34595 retention=69.2%
[omnisql] total=55000 converted=38049 retention=69.2%
[omnisql] total=60000 converted=41479 retention=69.1%
[omnisql] total=65000 converted=44884 retention=69.1%
[omnisql] total=70000 converted=48345 retention=69.1%
[omnisql] total=75000 converted=51820 retention=69.1%
[omnisql] total=80000 converted=55278 retention=69.1%
[omnisql] total=85000 converted=58713 retention=69.1%
[omnisql] total=95000 converted=65524 retention=69.0%
[omnisql] total=100000 converted=68977 retention=69.0%
gretel: 68977/100000 retention=69.0%
[omnisql-bird] total=6000 converted=4213 retention=70.2%
omnisql-bird: 6461/9428 retention=68.5%


In [22]:
# Concatenate into ultimate corpus.
!cat data/skeleton_train_v3_full.jsonl data/skeleton_train_v3_nstext2sql.jsonl data/skeleton_train_v3_gretel.jsonl data/skeleton_train_v3_omnisql_bird.jsonl > data/skeleton_train_v3_ultimate.jsonl
!wc -l data/skeleton_train_v3_ultimate.jsonl
!grep -c '"INNER JOIN"' data/skeleton_train_v3_ultimate.jsonl
!grep -c '"kind": "fk"' data/skeleton_train_v3_ultimate.jsonl

496104 data/skeleton_train_v3_ultimate.jsonl
0
176908


## 4. Derive Stage 1 + Stage 3 corpora

In [23]:
# Stage 1 multi-entity linker pairs
!python -m semsql_train derive-linker-pairs \
  --in data/skeleton_train_v3_ultimate.jsonl \
  --out data/linker_train_v3.jsonl \
  --max-rows 200000

# Split eval
import random, json
random.seed(7)
with open('data/linker_train_v3.jsonl') as f: lines = f.readlines()
random.shuffle(lines)
with open('data/linker_eval_v3.jsonl', 'w') as f: f.writelines(lines[:8000])
with open('data/linker_train_v3.jsonl', 'w') as f: f.writelines(lines[8000:])
print(f'linker train={len(lines)-8000} eval=8000')

# Stage 3 slot filler pairs (with rich extractor logic)
!python -m semsql_train derive-slot-pairs \
  --in data/skeleton_train_v3_ultimate.jsonl \
  --out data/slot_train_v3.jsonl \
  --max-rows 200000

with open('data/slot_train_v3.jsonl') as f: lines = f.readlines()
random.shuffle(lines)
with open('data/slot_eval_v3.jsonl', 'w') as f: f.writelines(lines[:8000])
with open('data/slot_train_v3.jsonl', 'w') as f: f.writelines(lines[8000:])
print(f'slot train={len(lines)-8000} eval=8000')

wrote 200000 Stage 1 pairs to data/linker_train_v3.jsonl
linker train=192000 eval=8000
wrote 200000 Stage 3 pairs to data/slot_train_v3.jsonl
slot train=192000 eval=8000


## 5. Train Stage 1 (linker, ~30-60 min T4)

In [ ]:
!python -m semsql_train train \
  --stage linker \
  --base-model distilbert-base-uncased \
  --train data/linker_train_v3.jsonl \
  --eval data/linker_eval_v3.jsonl \
  --epochs 3 \
  --batch-size 64 \
  --out target/linker-v3-gpu

preflight ok: train=192000 eval=8000 positive_fraction=53.18%
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
2026-05-09 06:07:20.365451: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-09 06:07:22.530747: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 270kB/s]
config.json: 100% 483/483 [00:00<00:00, 2.81MB/s]
vocab.txt: 232kB [00:00, 1.90MB/s]
tokenizer.json: 466kB [00:00, 2.28MB/s]
model.safetensors: 100% 268M/268M [00:02<

## 6. Train Stage 3 (slot filler, ~30-60 min T4)

In [1]:
!python -m semsql_train train \
  --stage slot-filler \
  --base-model distilbert-base-uncased \
  --train data/slot_train_v3.jsonl \
  --eval data/slot_eval_v3.jsonl \
  --epochs 3 \
  --batch-size 64 \
  --out target/slot-filler-v3-gpu

/usr/bin/python3: No module named semsql_train


## 7. Train Stage 2 (skeleton, biggest job)

Use t5-small Ã— 3 epochs on T4 (~6-8hr) or t5-base Ã— 5 epochs on A100 (~12-24hr).

For T4, the 50K JOIN-balanced subset finishes in ~1-2hr.

In [3]:
# Build JOIN-balanced 50K subset.
import random
import os

input_path = 'data/skeleton_train_v3_ultimate.jsonl'
output_path = 'data/skeleton_train_v3_50k_balanced.jsonl'

if not os.path.exists(input_path):
    print(f"Error: {input_path} not found. Ensure the concatenation step (cell 7RMFr57FJWZB) ran successfully.")
else:
    random.seed(0xCAFEF00D)
    with open(input_path, 'r') as f:
        lines = f.readlines()

    join_lines = [l for l in lines if 'INNER JOIN' in l]
    nojoin_lines = [l for l in lines if 'INNER JOIN' not in l]

    print(f"Total lines: {len(lines)} (JOINs: {len(join_lines)}, No-JOINs: {len(nojoin_lines)})")

    random.shuffle(join_lines)
    random.shuffle(nojoin_lines)

    # Take up to 25k of each to balance
    sub = join_lines[:25000] + nojoin_lines[:25000]
    random.shuffle(sub)

    with open(output_path, 'w') as f:
        f.writelines(sub)

    print(f'Created 50K subset: {sum(1 for l in sub if "INNER JOIN" in l)} JOINs')

Error: data/skeleton_train_v3_ultimate.jsonl not found. Ensure the concatenation step (cell 7RMFr57FJWZB) ran successfully.


In [4]:
import os
paths = [
    'data/skeleton_train_v3_full.jsonl',
    'data/skeleton_train_v3_nstext2sql.jsonl',
    'data/skeleton_train_v3_gretel.jsonl',
    'data/skeleton_train_v3_omnisql_bird.jsonl'
]
for p in paths:
    print(f"{p}: {'Exists' if os.path.exists(p) else 'MISSING'}")

data/skeleton_train_v3_full.jsonl: MISSING
data/skeleton_train_v3_nstext2sql.jsonl: MISSING
data/skeleton_train_v3_gretel.jsonl: MISSING
data/skeleton_train_v3_omnisql_bird.jsonl: MISSING


In [5]:
# T4 path (default). Switch --base-model to t5-base on A100.
!python -m semsql_train train \
  --stage skeleton \
  --base-model t5-small \
  --train data/skeleton_train_v3_50k_balanced.jsonl \
  --eval data/skeleton_eval.jsonl \
  --epochs 3 \
  --batch-size 16 \
  --grad-accum 2 \
  --out target/skeleton-v3-gpu

/usr/bin/python3: No module named semsql_train


## 8. Export cascade + tarball

In [6]:
import os
import json
from google.colab import files

# Ensure the source path is in PYTHONPATH for the shell command
%env PYTHONPATH=$PYTHONPATH:/content/db-claw/python/semsql_train/src

# Run the export command
!python -m semsql_train export-cascade \
  --output-dir target/cascade-v3-gpu \
  --cascade-version v0.3.0-gpu \
  --skeleton-checkpoint target/skeleton-v3-gpu \
  --linker-checkpoint target/linker-v3-gpu \
  --slot-filler-checkpoint target/slot-filler-v3-gpu

# Patch manifest only if the export succeeded
manifest_path = 'target/cascade-v3-gpu/manifest.json'
if os.path.exists(manifest_path):
    with open(manifest_path, 'r') as f:
        m = json.load(f)

    m['skeleton']['path'] = '_skeleton_export'
    m['skeleton']['tokenizer'] = '_skeleton_export/tokenizer.json'
    m['linker']['path'] = '_linker_export/model_quantized.onnx'
    m['linker']['tokenizer'] = '_linker_export/tokenizer.json'
    m['slot_filler']['path'] = '_slot_filler_export/model_quantized.onnx'
    m['slot_filler']['tokenizer'] = '_slot_filler_export/tokenizer.json'

    with open(manifest_path, 'w') as f:
        json.dump(m, f, indent=2)

    !tar czf cascade-v3-gpu.tar.gz -C target cascade-v3-gpu
    !ls -la cascade-v3-gpu.tar.gz
    files.download('cascade-v3-gpu.tar.gz')
else:
    print(f"Error: {manifest_path} was not created. Check if the training checkpoints exist in the 'target/' directory.")

/usr/bin/python3: No module named semsql_train


FileNotFoundError: [Errno 2] No such file or directory: 'target/cascade-v3-gpu/manifest.json'

## 9. (Optional) Local BIRD smoke

Run on your laptop after extracting the tarball. Requires `target/release/semsql.exe` built with `cargo build --release --features onnx -p semsql-cli`.

```bash
tar xzf cascade-v3-gpu.tar.gz -C target/
cp .venv/Lib/site-packages/onnxruntime/capi/onnxruntime.dll target/release/
rm -rf target/spider_graphs/*.semsql
uv run --package semsql-eval python -m semsql_eval spider \
  --questions data/bird/dev.json \
  --db-root data/bird/dev_databases \
  --name bird --limit 100 \
  --semsql-bin target/release/semsql.exe \
  --cascade-manifest target/cascade-v3-gpu/manifest.json \
  --report-json docs/results/v3-gpu-bird100-report.json \
  --graph-cache-dir target/spider_graphs
```

Acceptance gate per `docs/results/phase-d-gpu-handoff.md`:

- t5-small Ã— 3 epochs (T4): expect EX 5-15%
- t5-base Ã— 5 epochs (A100): expect EX 25-35%